<a href="https://colab.research.google.com/github/KaraIbr/SmartParkingGU/blob/main/notebooks/03_finetune_cnrpark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 - Fine-Tuning y Optimizacion

**Fase:** Fine-Tuning y Optimizacion (30% de la rubrica)

**Objetivo:**
- Implementar busqueda sistematica de hiperparametros
- Experimentar con diferentes configuraciones
- Superar las metricas del modelo base
- Documentar mejoras comprobables

**Dataset:** CNRPark-Patches-150x150
**Modelo base:** SmartParkingCNN

## 1. Configuracion e Imports

In [ ]:
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import random

# Agregar scripts al path
sys.path.append(os.path.join(os.getcwd(), 'scripts'))

from models import SmartParkingCNN, get_model_info
from train import train_model, load_checkpoint
from evaluate import (
    evaluate_model,
    plot_confusion_matrix,
    plot_roc_curve,
    plot_pr_curve,
    plot_training_curves,
    print_metrics,
    compare_models
)
from hyperparameter_search import (
    grid_search,
    random_search,
    save_results_to_csv,
    load_results_from_csv,
    find_best_config,
    set_seed
)

# Importar dataset del notebook anterior
sys.path.append(os.path.join(os.getcwd(), 'notebooks'))

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")

## 2. Cargar Dataset y Modelo Base

In [ ]:
# Configuracion
DATASET_ROOT = './dataset'
BATCH_SIZE = 32
IMG_SIZE = (150, 150)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Transformaciones
from torchvision import transforms
from PIL import Image

class ParkingDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir, split='entrenamiento', transform=None):
        self.root_dir = root_dir
        self.split = split
        self.transform = transform
        self.samples = []
        self.class_names = ['ocupado', 'no_ocupado']
        self.class_to_idx = {'ocupado': 0, 'no_ocupado': 1}
        
        for class_name in self.class_names:
            class_dir = os.path.join(root_dir, split, class_name)
            if os.path.exists(class_dir):
                for img_name in os.listdir(class_dir):
                    if img_name.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                        self.samples.append({
                            'path': os.path.join(class_dir, img_name),
                            'label': self.class_to_idx[class_name]
                        })
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        image = Image.open(sample['path']).convert('RGB')
        label = sample['label']
        if self.transform:
            image = self.transform(image)
        return image, label

# Transformaciones
val_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Crear datasets
train_dataset = ParkingDataset(DATASET_ROOT, split='entrenamiento', transform=val_transform)
val_dataset = ParkingDataset(DATASET_ROOT, split='validacion', transform=val_transform)
test_dataset = ParkingDataset(DATASET_ROOT, split='test', transform=val_transform)

# Crear dataloaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Dataset cargado:")
print(f"  Entrenamiento: {len(train_dataset)} imagenes")
print(f"  Validacion: {len(val_dataset)} imagenes")
print(f"  Prueba: {len(test_dataset)} imagenes")

In [ ]:
# Cargar metricas del modelo base para comparar
# Estas metricas vienen del notebook 02_base_model_cnrpark.ipynb
base_metrics = {
    'accuracy': 0.0,  # Se actualiza con los valores reales
    'precision_macro': 0.0,
    'recall_macro': 0.0,
    'f1_macro': 0.0
}

print("Metricas del modelo base (cargar de notebook 02):")
print(base_metrics)

## 3. Experimento 1: Grid Search

Busqueda sistematica sobre Learning Rate x Weight Decay

Segun el plan.md:
- Fase 1: Grid Search 4x4 (LR x Weight Decay) = 16 experimentos

In [ ]:
# Configurar Grid Search
param_grid = {
    'learning_rate': [1e-3, 5e-4, 1e-4, 5e-5],
    'weight_decay': [0, 1e-4, 1e-3, 5e-3],
    'dropout_rate': [0.5],  # Fijo para esta busqueda
    'batch_size': [32]  # Fijo
}

print("Parametros para Grid Search:")
for key, values in param_grid.items():
    print(f"  {key}: {values}")

total_combinations = 1
for values in param_grid.values():
    total_combinations *= len(values)
print(f"\nTotal de combinaciones: {total_combinations}")

In [ ]:
# Ejecutar Grid Search
grid_results = grid_search(
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    param_grid=param_grid,
    num_epochs=30,
    device=device,
    save_dir='experiments/cnrpark'
)

In [ ]:
# Guardar resultados del Grid Search
save_results_to_csv(grid_results, 'experiments/cnrpark/grid_search_results.csv')

# Mostrar top 5 resultados
grid_df = pd.DataFrame(grid_results)
print("\nTop 5 experimentos por F1-Score:")
print(grid_df.nlargest(5, 'f1_score')[['experiment_id', 'learning_rate', 'weight_decay', 
                                       'accuracy', 'f1_score']].to_string(index=False))

## 4. Experimento 2: Random Search

Busqueda aleatoria en espacios de parametros mas amplios

Segun el plan.md:
- Fase 2: Random Search = 15 experimentos adicionales

In [ ]:
# Configurar Random Search
param_distributions = {
    'learning_rate': [1e-5, 5e-5, 1e-4, 5e-4, 1e-3, 5e-3, 1e-2],
    'weight_decay': [0, 1e-5, 1e-4, 1e-3, 5e-3, 1e-2],
    'dropout_rate': [0.3, 0.4, 0.5, 0.6, 0.7],
    'batch_size': [16, 32, 64]
}

print("Distribuciones para Random Search:")
for key, values in param_distributions.items():
    print(f"  {key}: {values}")

In [ ]:
# Ejecutar Random Search
random_results = random_search(
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    param_distributions=param_distributions,
    n_iter=15,
    num_epochs=30,
    device=device,
    save_dir='experiments/cnrpark'
)

In [ ]:
# Guardar resultados del Random Search
save_results_to_csv(random_results, 'experiments/cnrpark/random_search_results.csv')

# Mostrar top 5 resultados
random_df = pd.DataFrame(random_results)
print("\nTop 5 experimentos Random Search por F1-Score:")
print(random_df.nlargest(5, 'f1_score')[['experiment_id', 'learning_rate', 'weight_decay', 
                                        'dropout_rate', 'accuracy', 'f1_score']].to_string(index=False))

## 5. Combinar y Analizar Resultados

In [ ]:
# Combinar todos los resultados
all_results = grid_results + random_results
all_df = pd.DataFrame(all_results)

# Guardar todos los resultados
all_df.to_csv('experiments/cnrpark/results.csv', index=False)

print(f"Total de experimentos: {len(all_results)}")
print(f"\nEstadisticas de F1-Score:")
print(f"  Minimo:  {all_df['f1_score'].min()*100:.2f}%")
print(f"  Maximo:  {all_df['f1_score'].max()*100:.2f}%")
print(f"  Promedio: {all_df['f1_score'].mean()*100:.2f}%")
print(f"  Mediana: {all_df['f1_score'].median()*100:.2f}%")

In [ ]:
# Top 10 mejores experimentos
print("\nTop 10 experimentos por F1-Score:")
top10 = all_df.nlargest(10, 'f1_score')
print(top10[['experiment_id', 'learning_rate', 'weight_decay', 'dropout_rate',
             'accuracy', 'precision', 'recall', 'f1_score']].to_string(index=False))

## 6. Analisis de Importancia de Parametros

In [ ]:
# Heatmap: Learning Rate vs Weight Decay
pivot_table = all_df.pivot_table(
    values='f1_score',
    index='weight_decay',
    columns='learning_rate',
    aggfunc='mean'
)

plt.figure(figsize=(10, 6))
sns.heatmap(pivot_table, annot=True, fmt='.4f', cmap='YlOrRd',
            xticklabels=['1e-5', '5e-5', '1e-4', '5e-4', '1e-3', '5e-3', '1e-2'],
            yticklabels=['0', '1e-5', '1e-4', '1e-3', '5e-3', '1e-2'])
plt.title('Heatmap: Learning Rate vs Weight Decay (F1-Score)', fontsize=14, fontweight='bold')
plt.xlabel('Learning Rate', fontsize=12)
plt.ylabel('Weight Decay', fontsize=12)
plt.tight_layout()
plt.savefig('reports/figures/heatmap_lr_wd.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Analisis de Dropout
if 'dropout_rate' in all_df.columns:
    dropout_stats = all_df.groupby('dropout_rate')['f1_score'].agg(['mean', 'std', 'max'])
    print("\nRendimiento por Dropout Rate:")
    print(dropout_stats.to_string())
    
    plt.figure(figsize=(8, 5))
    all_df.boxplot(column='f1_score', by='dropout_rate', figsize=(8, 5))
    plt.title('F1-Score por Dropout Rate', fontsize=14, fontweight='bold')
    plt.xlabel('Dropout Rate', fontsize=12)
    plt.ylabel('F1-Score', fontsize=12)
    plt.tight_layout()
    plt.savefig('reports/figures/boxplot_dropout.png', dpi=150, bbox_inches='tight')
    plt.show()

## 7. Entrenar Mejor Modelo

Entrenar el modelo con la mejor configuracion encontrada

In [ ]:
# Encontrar mejor configuracion
best_config = find_best_config(all_results, metric='f1_score')

print("Mejor configuracion encontrada:")
print(f"  Learning Rate:  {best_config['learning_rate']}")
print(f"  Weight Decay:   {best_config['weight_decay']}")
print(f"  Dropout Rate:   {best_config.get('dropout_rate', 0.5)}")
print(f"  Batch Size:     {best_config.get('batch_size', 32)}")
print(f"  F1-Score:       {best_config['f1_score']*100:.2f}%")
print(f"  Accuracy:       {best_config['accuracy']*100:.2f}%")

In [ ]:
# Entrenar modelo final con mejores hiperparametros
set_seed(42)

model_tuned = SmartParkingCNN(
    num_classes=2,
    dropout_rate=best_config.get('dropout_rate', 0.5)
).to(device)

history_tuned = train_model(
    model=model_tuned,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=60,
    learning_rate=best_config['learning_rate'],
    weight_decay=best_config['weight_decay'],
    device=device,
    save_dir='experiments/cnrpark/best_model',
    patience=15
)

In [ ]:
# Visualizar curvas del modelo optimizado
plot_training_curves(
    history_tuned,
    save_path='reports/figures/training_curves_tuned.png',
    title='Curvas de Entrenamiento - Modelo Optimizado'
)

## 8. Evaluar Modelo Optimizado

In [ ]:
# Cargar mejor modelo optimizado
model_final = SmartParkingCNN(
    num_classes=2,
    dropout_rate=best_config.get('dropout_rate', 0.5)
)
model_final, checkpoint_final = load_checkpoint(
    model_final,
    'experiments/cnrpark/best_model/best_model.pt'
)
model_final = model_final.to(device)

print(f"Modelo optimizado cargado desde epoca {checkpoint_final.get('epoch', 'N/A')}")

In [ ]:
# Evaluar en test set
metrics_tuned, preds_tuned, labels_tuned, probs_tuned = evaluate_model(
    model_final,
    test_loader,
    device,
    class_names=['Ocupado', 'Libre']
)

# Mostrar metricas
print_metrics(metrics_tuned, "METRICAS MODELO OPTIMIZADO - TEST SET")

In [ ]:
# Matriz de confusion
cm_tuned = plot_confusion_matrix(
    labels_tuned,
    preds_tuned,
    class_names=['Ocupado', 'Libre'],
    save_path='reports/figures/confusion_matrix_tuned.png',
    title='Matriz de Confusion - Modelo Optimizado'
)

In [ ]:
# ROC Curve
fpr_tuned, tpr_tuned, roc_auc_tuned = plot_roc_curve(
    labels_tuned,
    probs_tuned,
    class_names=['Ocupado', 'Libre'],
    save_path='reports/figures/roc_curve_tuned.png',
    title='ROC Curve - Modelo Optimizado'
)
print(f"\nROC-AUC Score: {roc_auc_tuned:.4f}")

In [ ]:
# PR Curve
precision_pr_tuned, recall_pr_tuned, pr_auc_tuned = plot_pr_curve(
    labels_tuned,
    probs_tuned,
    class_names=['Ocupado', 'Libre'],
    save_path='reports/figures/pr_curve_tuned.png',
    title='Precision-Recall Curve - Modelo Optimizado'
)
print(f"\nPR-AUC Score: {pr_auc_tuned:.4f}")

## 9. Comparativa: Base vs Optimizado

In [ ]:
# Comparar modelos
compare_models(
    base_metrics,
    metrics_tuned,
    model_names=['Modelo Base', 'Modelo Optimizado']
)

In [ ]:
# Tabla comparativa de metricas
comparison_df = pd.DataFrame({
    'Metrica': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'],
    'Modelo Base': [
        base_metrics['accuracy'],
        base_metrics['precision_macro'],
        base_metrics['recall_macro'],
        base_metrics['f1_macro'],
        0.0  # Cargar del notebook anterior
    ],
    'Modelo Optimizado': [
        metrics_tuned['accuracy'],
        metrics_tuned['precision_macro'],
        metrics_tuned['recall_macro'],
        metrics_tuned['f1_macro'],
        roc_auc_tuned
    ]
})

comparison_df['Mejora'] = comparison_df['Modelo Optimizado'] - comparison_df['Modelo Base']
comparison_df['Mejora %'] = (comparison_df['Mejora'] / comparison_df['Modelo Base'] * 100)

print("\nTabla Comparativa de Metricas:")
print(comparison_df.to_string(index=False, float_format='%.4f'))

## 10. Resumen Final

In [ ]:
# Resumen final
print("=" * 70)
print("RESUMEN FINAL - FINE-TUNING Y OPTIMIZACION")
print("=" * 70)

print(f"\nTotal de experimentos realizados: {len(all_results)}")
print(f"  - Grid Search: {len(grid_results)}")
print(f"  - Random Search: {len(random_results)}")

print(f"\nMejor configuracion encontrada:")
print(f"  Learning Rate: {best_config['learning_rate']}")
print(f"  Weight Decay:  {best_config['weight_decay']}")
print(f"  Dropout Rate:  {best_config.get('dropout_rate', 0.5)}")

print(f"\nMetricas finales (Test Set):")
print(f"  Accuracy:  {metrics_tuned['accuracy']*100:.2f}%")
print(f"  Precision: {metrics_tuned['precision_macro']*100:.2f}%")
print(f"  Recall:    {metrics_tuned['recall_macro']*100:.2f}%")
print(f"  F1-Score:  {metrics_tuned['f1_macro']*100:.2f}%")
print(f"  ROC-AUC:   {roc_auc_tuned:.4f}")

print(f"\nMejora vs Modelo Base:")
for metric_name, base_val, tuned_val in [
    ('Accuracy', base_metrics['accuracy'], metrics_tuned['accuracy']),
    ('F1-Score', base_metrics['f1_macro'], metrics_tuned['f1_macro'])
]:
    improvement = (tuned_val - base_val) * 100
    print(f"  {metric_name}: {base_val*100:.2f}% -> {tuned_val*100:.2f}% ({improvement:+.2f}%)")

print(f"\nArchivos generados:")
print(f"  - experiments/cnrpark/results.csv")
print(f"  - experiments/cnrpark/best_model/best_model.pt")
print(f"  - reports/figures/ (graficos)")

print("=" * 70)

## 11. Siguiente Paso

El modelo optimizado ha sido entrenado y evaluado. Ahora procederemos a:

1. **Exportacion del Modelo** en el notebook `04_export_model.ipynb`
2. Serializar el modelo en formato `.pt`
3. Crear script de inferencia